# 🖼️ Image Captioning với BLIP trên Flickr30k

**Pipeline đầy đủ:**
- ✅ Load & khám phá Flickr30k dataset
- ✅ Tiền xử lý ảnh & caption
- ✅ Fine-tune BLIP với **chiến lược Freeze/Unfreeze 2 giai đoạn**
- ✅ Đánh giá bằng BLEU score
- ✅ Demo sinh caption cho ảnh mới

## 🧊 Chiến lược Freeze/Unfreeze

| Giai đoạn | Epoch | Vision Encoder | Text Decoder | Mục đích |
|-----------|-------|---------------|--------------|----------|
| **Phase 1** | 1 → 3 | ❄️ Frozen | 🔥 Train | Học caption nhanh, ít VRAM |
| **Phase 2** | 4 → 5 | 🔥 Unfreeze | 🔥 Train | Fine-tune toàn bộ, lr thấp hơn |

> ⚠️ **Lưu ý:** Đảm bảo Runtime → Change runtime type → **GPU (T4)**

## 📦 1. Cài đặt thư viện

In [ ]:
!pip install -q transformers datasets evaluate nltk Pillow torch torchvision tqdm pycocoevalcap
!pip install -q accelerate
print('✅ Cài đặt hoàn tất!')

## 📚 2. Import thư viện

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration,
    get_linear_schedule_with_warmup
)

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📂 3. Mount Google Drive & Cấu hình đường dẫn

Đảm bảo Flickr30k của bạn có cấu trúc:
```
flickr30k/
├── images/          # ~31.000 ảnh .jpg
└── captions.txt     # hoặc results.csv / annotations.token
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Chỉnh đường dẫn theo vị trí dataset của bạn ──────────
DATASET_ROOT = Path('/content/drive/MyDrive/flickr30k')   # ← SỬA ĐÂY
IMAGES_DIR   = DATASET_ROOT / 'images'
CAPTIONS_FILE = DATASET_ROOT / 'captions.txt'             # ← SỬA ĐÂY nếu tên file khác

# Thư mục lưu checkpoint
CHECKPOINT_DIR = Path('/content/drive/MyDrive/blip_flickr30k_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'📁 Dataset : {DATASET_ROOT}')
print(f'🖼️  Ảnh     : {IMAGES_DIR}')
print(f'📝 Captions: {CAPTIONS_FILE}')
print(f'💾 Checkpoint: {CHECKPOINT_DIR}')

## 🔍 4. Load & Khám phá Dataset

In [ ]:
def load_flickr30k_captions(captions_file: Path) -> pd.DataFrame:
    """
    Hỗ trợ nhiều định dạng caption phổ biến của Flickr30k:
    1. captions.txt  : image_name|caption_number|caption
    2. results.csv   : image_name,caption_number,caption
    3. .token file   : image_name#caption_number\tcaption
    """
    rows = []
    ext = captions_file.suffix.lower()

    with open(captions_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('image'):  # bỏ header
                continue

            if '|' in line:                           # format 1
                parts = line.split('|')
                img_name, cap_idx, caption = parts[0], parts[1], '|'.join(parts[2:])
            elif '\t' in line:                        # format 3
                left, caption = line.split('\t', 1)
                img_name, cap_idx = left.split('#')
            else:                                     # format 2 (csv)
                parts = line.split(',', 2)
                img_name, cap_idx, caption = parts[0], parts[1], parts[2]

            rows.append({
                'image_name': img_name.strip(),
                'caption_idx': int(str(cap_idx).strip()),
                'caption': caption.strip()
            })

    return pd.DataFrame(rows)


df = load_flickr30k_captions(CAPTIONS_FILE)
print(f'✅ Tổng số dòng      : {len(df):,}')
print(f'   Số ảnh unique     : {df["image_name"].nunique():,}')
print(f'   Captions/ảnh      : {len(df) / df["image_name"].nunique():.1f}')
print()
df.head(10)

In [ ]:
# ── Thống kê caption ─────────────────────────────────────
df['caption_len'] = df['caption'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['caption_len'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Phân phối độ dài Caption (số từ)', fontsize=13)
axes[0].set_xlabel('Số từ'); axes[0].set_ylabel('Số lượng')
axes[0].axvline(df['caption_len'].mean(), color='red', linestyle='--',
                label=f'Mean={df["caption_len"].mean():.1f}')
axes[0].legend()

caps_per_img = df.groupby('image_name').size()
axes[1].bar(caps_per_img.value_counts().index, caps_per_img.value_counts().values,
            color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Số Caption mỗi ảnh', fontsize=13)
axes[1].set_xlabel('Số captions'); axes[1].set_ylabel('Số ảnh')

plt.tight_layout()
plt.show()

print(f'Caption ngắn nhất : {df["caption_len"].min()} từ')
print(f'Caption dài nhất  : {df["caption_len"].max()} từ')
print(f'Trung bình        : {df["caption_len"].mean():.1f} từ')

In [ ]:
# ── Hiển thị một số ảnh mẫu ──────────────────────────────
sample_imgs = df['image_name'].drop_duplicates().sample(6, random_state=SEED).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_name in zip(axes.flat, sample_imgs):
    img_path = IMAGES_DIR / img_name
    img = Image.open(img_path).convert('RGB')
    captions = df[df['image_name'] == img_name]['caption'].tolist()
    ax.imshow(img)
    ax.set_title('\n'.join([f'{i+1}. {c}' for i, c in enumerate(captions[:2])]),
                 fontsize=8, loc='left', pad=6)
    ax.axis('off')

plt.suptitle('Mẫu ảnh & Caption từ Flickr30k', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## ✂️ 5. Chia Train / Validation / Test

In [ ]:
# Chia theo image (không bị data leakage) — tỉ lệ 8:1:1
all_images = df['image_name'].drop_duplicates().tolist()
random.shuffle(all_images)

n_total = len(all_images)
n_train = int(n_total * 0.8)
n_val   = int(n_total * 0.1)
n_test  = n_total - n_train - n_val

train_imgs = set(all_images[:n_train])
val_imgs   = set(all_images[n_train:n_train + n_val])
test_imgs  = set(all_images[n_train + n_val:])

train_df = df[df['image_name'].isin(train_imgs)].reset_index(drop=True)
val_df   = df[df['image_name'].isin(val_imgs)].reset_index(drop=True)
test_df  = df[df['image_name'].isin(test_imgs)].reset_index(drop=True)

print(f'Tổng  : {n_total:,} ảnh')
print(f'Train : {len(train_imgs):,} ảnh ({len(train_imgs)/n_total*100:.0f}%) | {len(train_df):,} captions')
print(f'Val   : {len(val_imgs):,} ảnh ({len(val_imgs)/n_total*100:.0f}%) | {len(val_df):,} captions')
print(f'Test  : {len(test_imgs):,} ảnh ({len(test_imgs)/n_total*100:.0f}%) | {len(test_df):,} captions')

## 🏗️ 6. Dataset & DataLoader

In [ ]:
# ── Config ───────────────────────────────────────────────
CFG = {
    'model_name'       : 'Salesforce/blip-image-captioning-base',
    'max_length'       : 40,   # ✅ Opt-4: giảm từ 64→40 (Flickr30k avg ~12 từ)
    'batch_size'       : 8,    # ✅ Opt-1: batch nhỏ, bù bằng grad accumulation
    'grad_accum_steps' : 4,    # ✅ Opt-1: 8×4 = batch ảo 32
    'num_workers'      : 2,
    # ── Freeze strategy ──────────────────────────────────
    'freeze_epochs'    : 3,
    'lr_phase1'        : 2e-4,
    'lr_phase2'        : 2e-5,
    # ── Early Stopping ───────────────────────────────────
    'patience'         : 3,    # dừng nếu val loss không giảm sau 3 epoch
    # ─────────────────────────────────────────────────────
    'weight_decay'     : 1e-2,
    'epochs'           : 10,   # tăng lên 10, Early Stopping tự dừng đúng lúc
    'warmup_ratio'     : 0.1,
    'grad_clip'        : 1.0,
    'log_interval'     : 100,
    'save_every'       : 1,
}


class Flickr30kDataset(Dataset):
    """Dataset trả về (pixel_values, input_ids, attention_mask)."""

    def __init__(self, dataframe: pd.DataFrame, images_dir: Path,
                 processor: BlipProcessor, max_length: int = 64):
        # Mỗi hàng là 1 (ảnh, caption) pair — ảnh có thể lặp lại
        self.data       = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row     = self.data.iloc[idx]
        img_path = self.images_dir / row['image_name']

        # Load ảnh
        image = Image.open(img_path).convert('RGB')

        # Tiền xử lý: ảnh + caption cùng lúc
        encoding = self.processor(
            images=image,
            text=row['caption'],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        pixel_values   = encoding['pixel_values'].squeeze(0)   # (3, H, W)
        input_ids      = encoding['input_ids'].squeeze(0)       # (L,)
        attention_mask = encoding['attention_mask'].squeeze(0)  # (L,)

        # Labels: -100 tại vị trí padding để bỏ qua khi tính loss
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        return {
            'pixel_values'  : pixel_values,
            'input_ids'     : input_ids,
            'attention_mask': attention_mask,
            'labels'        : labels,
        }

In [ ]:
# ── Khởi tạo Processor & Datasets ───────────────────────
print('⏳ Load BLIP Processor...')
processor = BlipProcessor.from_pretrained(CFG['model_name'])

train_dataset = Flickr30kDataset(train_df, IMAGES_DIR, processor, CFG['max_length'])
val_dataset   = Flickr30kDataset(val_df,   IMAGES_DIR, processor, CFG['max_length'])
test_dataset  = Flickr30kDataset(test_df,  IMAGES_DIR, processor, CFG['max_length'])

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

print(f'✅ Train batches: {len(train_loader):,}')
print(f'   Val batches  : {len(val_loader):,}')
print(f'   Test batches : {len(test_loader):,}')

# Kiểm tra 1 batch
batch = next(iter(train_loader))
print(f'\n📐 Shapes:')
for k, v in batch.items():
    print(f'   {k}: {v.shape}')

## 🤖 7. Khởi tạo Model BLIP

In [ ]:
print('⏳ Load BLIP model...')
model = BlipForConditionalGeneration.from_pretrained(CFG['model_name'])

# ✅ Opt-3: torch.compile — tối ưu computation graph (~20% nhanh hơn)
# Lần đầu compile mất ~3 phút, các epoch sau nhanh hơn
print('⚙️  Compiling model (torch.compile)... ~3 phút lần đầu')
model = torch.compile(model)

model = model.to(DEVICE)

# ── Hàm tiện ích Freeze / Unfreeze ───────────────────────
def freeze_vision_encoder(model):
    """Đóng băng toàn bộ vision encoder (ViT)."""
    for param in model.vision_model.parameters():
        param.requires_grad = False
    # Đảm bảo text decoder vẫn trainable
    for param in model.text_decoder.parameters():
        param.requires_grad = True

def unfreeze_all(model):
    """Mở khóa toàn bộ tham số model."""
    for param in model.parameters():
        param.requires_grad = True

def count_trainable(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# ── Bắt đầu Phase 1: Freeze vision encoder ───────────────
freeze_vision_encoder(model)
total, trainable = count_trainable(model)

print(f'\n📊 Tổng tham số        : {total/1e6:.1f}M')
print(f'   Trainable (Phase 1) : {trainable/1e6:.1f}M  (text decoder only)')
print(f'   Frozen              : {(total-trainable)/1e6:.1f}M  (vision encoder)')
print(f'   VRAM hiện tại       : {torch.cuda.memory_allocated()/1e9:.2f} GB')

## ⚙️ 8. Optimizer, Scheduler & Mixed Precision

In [ ]:
# ── Hàm tạo optimizer theo phase ─────────────────────────
no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']

def build_optimizer(model, lr):
    """Tạo AdamW chỉ với các tham số đang requires_grad."""
    param_groups = [
        {'params': [p for n, p in model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         'weight_decay': CFG['weight_decay']},
        {'params': [p for n, p in model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    return AdamW(param_groups, lr=lr)

def build_scheduler(optimizer, num_epochs, warmup_ratio=0.1):
    total_steps  = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    return get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    ), total_steps, warmup_steps

# ── Khởi tạo Phase 1 (vision frozen) ─────────────────────
optimizer_p1 = build_optimizer(model, CFG['lr_phase1'])
scheduler_p1, steps_p1, warmup_p1 = build_scheduler(
    optimizer_p1, CFG['freeze_epochs'], CFG['warmup_ratio']
)

# AMP scaler dùng chung cả 2 phase
scaler = torch.cuda.amp.GradScaler()

print('✅ Phase 1 Optimizer:')
print(f'   lr={CFG["lr_phase1"]}  |  epochs={CFG["freeze_epochs"]}  |  '
      f'steps={steps_p1:,}  |  warmup={warmup_p1}')
print(f'\n   Phase 2 sẽ được tạo sau khi unfreeze ở epoch {CFG["freeze_epochs"]+1}')

## 🏋️ 9. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', leave=False)
    optimizer.zero_grad()                              # ✅ Opt-1: reset ở ngoài loop

    for step, batch in enumerate(pbar):
        pixel_values   = batch['pixel_values'].to(DEVICE)
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            # ✅ Opt-1: chia loss cho số bước tích lũy
            loss = outputs.loss / CFG['grad_accum_steps']

        scaler.scale(loss).backward()

        # ✅ Opt-1: chỉ update sau khi tích lũy đủ grad_accum_steps bước
        if (step + 1) % CFG['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG['grad_accum_steps']  # khôi phục loss gốc để log
        pbar.set_postfix({'loss': f'{loss.item() * CFG["grad_accum_steps"]:.4f}',
                          'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(model, loader, epoch):
    model.eval()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Val]', leave=False)

    for batch in pbar:
        pixel_values   = batch['pixel_values'].to(DEVICE)
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
        total_loss += outputs.loss.item()
        pbar.set_postfix({'val_loss': f'{outputs.loss.item():.4f}'})

    return total_loss / len(loader)

In [ ]:
history = {
    'epoch': [], 'phase': [],
    'train_loss': [], 'val_loss': [],
    'trainable_params': [], 'lr': []
}
best_val_loss    = float('inf')
patience_counter = 0                           # ✅ Early Stopping counter

# Dùng Phase 1 optimizer/scheduler ngay từ đầu
optimizer = optimizer_p1
scheduler = scheduler_p1

print('🚀 Bắt đầu training với chiến lược Freeze/Unfreeze + Early Stopping\n')
print(f'   Phase 1 (Frozen Vision): Epoch 1 → {CFG["freeze_epochs"]}  |  lr={CFG["lr_phase1"]}')
print(f'   Phase 2 (Full Finetune): Epoch {CFG["freeze_epochs"]+1} → {CFG["epochs"]}  |  lr={CFG["lr_phase2"]}')
print(f'   Early Stopping patience: {CFG["patience"]} epoch')
print()

for epoch in range(1, CFG['epochs'] + 1):

    # ── Chuyển phase ──────────────────────────────────────
    if epoch == CFG['freeze_epochs'] + 1:
        print('\n' + '='*60)
        print(f'🔓 PHASE 2: Unfreeze toàn bộ model tại Epoch {epoch}')
        unfreeze_all(model)
        total, trainable = count_trainable(model)
        print(f'   Trainable params: {trainable/1e6:.1f}M / {total/1e6:.1f}M')

        # Tạo optimizer & scheduler mới với lr thấp hơn
        optimizer = build_optimizer(model, CFG['lr_phase2'])
        remaining_epochs = CFG['epochs'] - CFG['freeze_epochs']
        scheduler, steps_p2, warmup_p2 = build_scheduler(
            optimizer, remaining_epochs, CFG['warmup_ratio']
        )
        print(f'   Optimizer mới: lr={CFG["lr_phase2"]}  |  steps={steps_p2:,}  |  warmup={warmup_p2}')
        print('='*60 + '\n')

    # Xác định phase hiện tại
    phase = 1 if epoch <= CFG['freeze_epochs'] else 2
    _, trainable = count_trainable(model)

    print(f'[Phase {phase}] Epoch {epoch}/{CFG["epochs"]}  |  '
          f'Trainable: {trainable/1e6:.1f}M params')

    # ── Train ─────────────────────────────────────────────
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    val_loss   = validate_epoch(model, val_loader, epoch)
    current_lr = scheduler.get_last_lr()[0]

    history['epoch'].append(epoch)
    history['phase'].append(phase)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['trainable_params'].append(trainable)
    history['lr'].append(current_lr)

    print(f'   Train Loss: {train_loss:.4f}  |  Val Loss: {val_loss:.4f}  |  lr: {current_lr:.2e}')

    # ── Checkpoint ────────────────────────────────────────
    if epoch % CFG['save_every'] == 0:
        ckpt_path = CHECKPOINT_DIR / f'epoch_{epoch:02d}_phase{phase}.pt'
        torch.save({
            'epoch'               : epoch,
            'phase'               : phase,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss'            : val_loss,
            'config'              : CFG,
        }, ckpt_path)
        print(f'   💾 Saved: {ckpt_path.name}')

    # ── Best model & Early Stopping ───────────────────────
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0                       # reset khi có cải thiện
        model.save_pretrained(CHECKPOINT_DIR / 'best_model')
        processor.save_pretrained(CHECKPOINT_DIR / 'best_model')
        print(f'   ⭐ Best model! Val Loss: {val_loss:.4f}')
    else:
        patience_counter += 1
        print(f'   ⏳ No improvement  '
              f'(patience {patience_counter}/{CFG["patience"]})  '
              f'| Best: {best_val_loss:.4f}')
        if patience_counter >= CFG['patience']:
            print(f'\n🛑 Early Stopping tại Epoch {epoch}!'
                  f' Val loss không cải thiện sau {CFG["patience"]} epoch.')
            print(f'   Best Val Loss đạt được: {best_val_loss:.4f}')
            break

print('\n✅ Training hoàn tất!')

In [ ]:
# ── Vẽ learning curves có đánh dấu phase ────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs_range = history['epoch']
phase_colors = ['steelblue' if p == 1 else 'darkorange' for p in history['phase']]

# ── Loss curves ───────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='steelblue', lw=2)
ax.plot(epochs_range, history['val_loss'],   's--', label='Val Loss',   color='coral',     lw=2)
ax.axvline(CFG['freeze_epochs'] + 0.5, color='gray', linestyle=':', lw=1.5,
           label=f'Unfreeze (Epoch {CFG["freeze_epochs"]+1})')
ax.fill_betweenx([0, max(history['train_loss'])*1.1],
                 0.5, CFG['freeze_epochs'] + 0.5,
                 alpha=0.06, color='steelblue', label='Phase 1 (frozen)')
ax.fill_betweenx([0, max(history['train_loss'])*1.1],
                 CFG['freeze_epochs'] + 0.5, CFG['epochs'] + 0.5,
                 alpha=0.06, color='darkorange', label='Phase 2 (full)')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Learning Curves — Freeze/Unfreeze', fontsize=13)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── LR schedule ───────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(epochs_range, history['lr'], color=phase_colors, alpha=0.8, width=0.6)
ax2.axvline(CFG['freeze_epochs'] + 0.5, color='gray', linestyle=':', lw=1.5)
ax2.set_xlabel('Epoch', fontsize=12); ax2.set_ylabel('Learning Rate', fontsize=12)
ax2.set_title('Learning Rate per Epoch', fontsize=13)
from matplotlib.patches import Patch
legend_els = [Patch(color='steelblue', alpha=0.8, label='Phase 1 (frozen vision)'),
              Patch(color='darkorange', alpha=0.8, label='Phase 2 (full finetune)')]
ax2.legend(handles=legend_els, fontsize=9); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 📏 10. Đánh giá: BLEU Score trên Test Set

In [ ]:
@torch.no_grad()
def generate_captions_batch(model, processor, image_paths: list,
                             max_new_tokens: int = 64) -> list:
    """Sinh caption cho một batch ảnh."""
    images = [Image.open(p).convert('RGB') for p in image_paths]
    inputs = processor(images=images, return_tensors='pt').to(DEVICE)

    with torch.cuda.amp.autocast():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return processor.batch_decode(out_ids, skip_special_tokens=True)


def compute_bleu(references_dict: dict, hypotheses_dict: dict):
    """
    references_dict : {img_name: [list_of_ref_captions]}
    hypotheses_dict : {img_name: generated_caption}
    Trả về BLEU-1,2,3,4
    """
    refs, hyps = [], []
    for img in hypotheses_dict:
        if img not in references_dict:
            continue
        ref_tokens  = [nltk.word_tokenize(r.lower()) for r in references_dict[img]]
        hyp_tokens  = nltk.word_tokenize(hypotheses_dict[img].lower())
        refs.append(ref_tokens)
        hyps.append(hyp_tokens)

    sf = SmoothingFunction().method1
    scores = {}
    for n in [1, 2, 3, 4]:
        weights = tuple([1/n] * n + [0] * (4 - n))
        scores[f'BLEU-{n}'] = corpus_bleu(refs, hyps, weights=weights,
                                            smoothing_function=sf)
    return scores

In [ ]:
# ── Load best model ───────────────────────────────────────
print('⏳ Load best model...')
best_model = BlipForConditionalGeneration.from_pretrained(
    CHECKPOINT_DIR / 'best_model'
).to(DEVICE)
best_model.eval()

# ── Sinh caption trên test set ────────────────────────────
EVAL_BATCH = 8
test_images = test_df['image_name'].drop_duplicates().tolist()

# References: image_name → list[str]
refs_dict = test_df.groupby('image_name')['caption'].apply(list).to_dict()

hyps_dict = {}
for i in tqdm(range(0, len(test_images), EVAL_BATCH), desc='Generating captions'):
    batch_names = test_images[i:i + EVAL_BATCH]
    batch_paths = [IMAGES_DIR / n for n in batch_names]
    captions    = generate_captions_batch(best_model, processor, batch_paths)
    for name, cap in zip(batch_names, captions):
        hyps_dict[name] = cap

# ── Tính BLEU ─────────────────────────────────────────────
bleu_scores = compute_bleu(refs_dict, hyps_dict)

print('\n📊 Kết quả BLEU trên Test Set:')
for k, v in bleu_scores.items():
    print(f'   {k}: {v:.4f}  ({v*100:.2f}%)')

In [ ]:
# ── Vẽ BLEU scores ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
names  = list(bleu_scores.keys())
values = [v * 100 for v in bleu_scores.values()]
bars = ax.bar(names, values, color=['#4C72B0','#DD8452','#55A868','#C44E52'], width=0.5)
ax.bar_label(bars, fmt='%.2f%%', padding=3, fontsize=11)
ax.set_ylim(0, max(values) * 1.3)
ax.set_ylabel('BLEU Score (%)', fontsize=12)
ax.set_title('BLEU Scores — BLIP on Flickr30k Test Set', fontsize=14)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 🎨 11. Demo: Sinh Caption cho Ảnh

In [ ]:
def demo_captions(model, processor, image_names: list,
                  images_dir: Path, refs_dict: dict, n: int = 6):
    """Hiển thị ảnh kèm caption sinh ra vs reference."""
    samples = random.sample(image_names, min(n, len(image_names)))
    paths   = [images_dir / s for s in samples]
    generated = generate_captions_batch(model, processor, paths)

    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    axes = axes.flat

    for ax, img_name, gen_cap in zip(axes, samples, generated):
        img = Image.open(images_dir / img_name).convert('RGB')
        ref = refs_dict.get(img_name, [''])[0]

        ax.imshow(img)
        ax.axis('off')
        title = (f'🤖 {gen_cap}\n\n'
                 f'📝 {ref}')
        ax.set_title(title, fontsize=8.5, loc='left', pad=6,
                     wrap=True)

    # Ẩn axes thừa
    for ax in axes:
        ax.axis('off')

    plt.suptitle('🤖 = Generated   |   📝 = Reference', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


demo_captions(best_model, processor, test_images, IMAGES_DIR, refs_dict)

## 🌐 12. Inference: Ảnh Tùy Chỉnh (Upload hoặc URL)

In [ ]:
import requests
from io import BytesIO

def caption_from_url(url: str) -> str:
    response = requests.get(url, timeout=10)
    image = Image.open(BytesIO(response.content)).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.cuda.amp.autocast():
        out = best_model.generate(**inputs, max_new_tokens=64, num_beams=4)
    caption = processor.decode(out[0], skip_special_tokens=True)

    # Hiển thị
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(image)
    ax.axis('off')
    ax.set_title(f'🤖 {caption}', fontsize=11, pad=10)
    plt.tight_layout()
    plt.show()
    return caption


# ── Thử với ảnh từ URL ────────────────────────────────────
TEST_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg'
cap = caption_from_url(TEST_URL)
print(f'Caption: {cap}')

In [ ]:
# ── Upload ảnh từ máy tính ────────────────────────────────
from google.colab import files

uploaded = files.upload()
for filename, data in uploaded.items():
    image = Image.open(BytesIO(data)).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.cuda.amp.autocast():
        out = best_model.generate(**inputs, max_new_tokens=64, num_beams=4)
    caption = processor.decode(out[0], skip_special_tokens=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(image); ax.axis('off')
    ax.set_title(f'🤖 {caption}', fontsize=11, pad=10)
    plt.tight_layout(); plt.show()
    print(f'Caption cho "{filename}": {caption}')

## 💾 13. Lưu & Load Model


In [ ]:
# ── Lưu toàn bộ model + processor ────────────────────────
FINAL_SAVE_PATH = CHECKPOINT_DIR / 'final_model'
best_model.save_pretrained(FINAL_SAVE_PATH)
processor.save_pretrained(FINAL_SAVE_PATH)
print(f'✅ Model saved to: {FINAL_SAVE_PATH}')

# ── Để load lại ───────────────────────────────────────────
# model = BlipForConditionalGeneration.from_pretrained(FINAL_SAVE_PATH)
# processor = BlipProcessor.from_pretrained(FINAL_SAVE_PATH)

---
## 📌 Tóm tắt Pipeline

| Bước | Chi tiết |
|------|----------|
| **Dataset** | Flickr30k (~31k ảnh, 5 captions/ảnh) |
| **Model** | BLIP `blip-image-captioning-base` (~247M params) |
| **Phase 1** | Epoch 1–3: Freeze vision encoder, chỉ train text decoder (lr=2e-4) |
| **Phase 2** | Epoch 4–5: Unfreeze toàn bộ, fine-tune nhẹ (lr=2e-5) |
| **Optimizer** | AdamW + Linear Warmup (riêng cho mỗi phase) |
| **AMP** | `torch.cuda.amp` — giảm VRAM, tăng tốc |
| **Evaluation** | BLEU-1/2/3/4 |
| **Inference** | URL, upload file, hoặc test set |

### 🔧 Gợi ý cải thiện
- Tăng `freeze_epochs` lên 4 nếu muốn ổn định hơn trước khi unfreeze
- Dùng **BLIP-2** (`Salesforce/blip2-opt-2.7b`) để cải thiện BLEU
- Thêm **CIDEr / METEOR** làm metric
- Dùng **gradient checkpointing** nếu OOM: `model.gradient_checkpointing_enable()`
- Layer-wise learning rate decay: vision encoder lr thấp hơn text decoder